# Assignment 3: Sequence Representations and Attention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/neural-architectures-and-representation-learning-course/blob/main/weeks/08/Assignment_03_Sequence_Representations_Attention.ipynb)

**Course:** Neural Architectures and Representation Learning  
**Related notebook:** `Week_08_Sequence_Representations_Attention.ipynb`


## Task

You will train and analyze small sequence models.

The goal is not to build the biggest model. The goal is to show evidence that you understand how sequence representations behave.

You will submit:

1. A baseline character-RNN run.
2. Two controlled character-RNN experiments.
3. Hidden-state or prefix-probability visualizations.
4. A tiny attention-model analysis on short sentences.
5. Attention heatmaps for at least three sentences.
6. Written interpretation of what the representations capture and miss.


## Grading rubric

| Criterion | Points |
|----------|--------|
| Correctly runs the baseline sequence model | 15 |
| Performs controlled character-RNN experiments | 20 |
| Uses hidden-state or prefix-probability visualizations | 20 |
| Analyzes word embeddings or attention behavior | 20 |
| Explains limitations and surprising examples | 15 |
| Clear, reproducible notebook | 10 |

Total: 100 points.


---

## Environment

This notebook uses `torch`, `numpy`, `matplotlib`, and `scikit-learn`. CPU is enough.

Do not change the fixed dataset cells unless an instruction explicitly says to add examples in an extension.

## Colab pointers

**RNN part:** run cells in order. The trained `baseline_model`, `experiment_1_model`, and `experiment_2_model` are reused by the visualization cells. In `CharRNNClassifier`, `embedding` maps characters to vectors, `rnn` updates the hidden memory, and `classifier` predicts from the final memory.

**Embedding/attention part:** the attention model also contains an embedding table at `attention_model.embedding.weight`. Use attention heatmaps and the optional embedding projection as inspection tools; custom sentences are easiest to interpret when they use words from the tiny vocabulary.


In [ ]:
import random
import re
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from sklearn.decomposition import PCA

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    plt.rc("axes", grid=True)

%matplotlib inline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(8)


---

## Part A: fixed names dataset

This is a tiny toy dataset for sequence representation analysis. Treat it as a controlled pattern-learning task, not as a serious demographic classifier.


In [ ]:
raw_names = {
    "English": ["Smith", "Johnson", "Williams", "Brown", "Jones", "Anderson", "Wilson", "Taylor", "Thomas", "Moore", "Martin", "Jackson", "Thompson", "White", "Harris", "Clark"],
    "Italian": ["Rossi", "Russo", "Ferrari", "Esposito", "Bianchi", "Romano", "Colombo", "Ricci", "Marino", "Greco", "Bruno", "Gallo", "Conti", "Deluca", "Costa", "Mancini"],
    "German": ["Muller", "Schmidt", "Schneider", "Fischer", "Weber", "Meyer", "Wagner", "Becker", "Schulz", "Hoffmann", "Schafer", "Koch", "Bauer", "Richter", "Klein", "Wolf"],
    "Slavic": ["Novak", "Kowalski", "Wisniewski", "Kaminski", "Lewandowski", "Zielinski", "Sokolov", "Popov", "Ivanov", "Petrov", "Volkov", "Morozov", "Smirnov", "Kuznetsov", "Horvat", "Kovacic"],
}

labels = list(raw_names.keys())
label_to_idx = {label: i for i, label in enumerate(labels)}
idx_to_label = {i: label for label, i in label_to_idx.items()}

all_examples = []
for label, names in raw_names.items():
    for name in names:
        all_examples.append((name.lower(), label_to_idx[label]))

random.shuffle(all_examples)
split = int(0.8 * len(all_examples))
train_examples = all_examples[:split]
test_examples = all_examples[split:]

chars = sorted(set("".join(name for name, _ in all_examples)))
char_to_idx = {"<PAD>": 0}
for ch in chars:
    char_to_idx[ch] = len(char_to_idx)
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

print("train/test:", len(train_examples), len(test_examples))
print("labels:", labels)


In [ ]:
def encode_name(name):
    return torch.tensor([char_to_idx[ch] for ch in name.lower()], dtype=torch.long)


def collate_names(batch):
    encoded = [encode_name(name) for name, _ in batch]
    lengths = torch.tensor([len(x) for x in encoded], dtype=torch.long)
    y = torch.tensor([label for _, label in batch], dtype=torch.long)
    max_len = int(lengths.max())
    x = torch.zeros(len(batch), max_len, dtype=torch.long)
    for i, item in enumerate(encoded):
        x[i, : len(item)] = item
    return x, lengths, y

train_loader = DataLoader(train_examples, batch_size=16, shuffle=True, collate_fn=collate_names)
test_loader = DataLoader(test_examples, batch_size=32, shuffle=False, collate_fn=collate_names)

class CharRNNClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, embedding_dim=12, hidden_dim=24):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed)
        return self.classifier(h_n[-1])

    @torch.no_grad()
    def hidden_path(self, name):
        self.eval()
        x = encode_name(name).unsqueeze(0).to(next(self.parameters()).device)
        outputs, _ = self.rnn(self.embedding(x))
        logits_each = self.classifier(outputs.squeeze(0))
        return outputs.squeeze(0).cpu(), logits_each.cpu()


def accuracy_from_logits(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()


def train_char_rnn(config):
    set_seed(config.get("seed", 8))
    model = CharRNNClassifier(
        len(char_to_idx), len(labels),
        embedding_dim=config.get("embedding_dim", 12),
        hidden_dim=config.get("hidden_dim", 24),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.get("lr", 0.02))
    history = {"loss": [], "train_acc": [], "test_acc": []}
    for epoch in range(config.get("epochs", 45)):
        model.train()
        losses, accs = [], []
        for x, lengths, y in train_loader:
            x, lengths, y = x.to(device), lengths.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x, lengths)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            accs.append(accuracy_from_logits(logits, y))
        test_accs = []
        model.eval()
        with torch.no_grad():
            for x, lengths, y in test_loader:
                x, lengths, y = x.to(device), lengths.to(device), y.to(device)
                test_accs.append(accuracy_from_logits(model(x, lengths), y))
        history["loss"].append(float(np.mean(losses)))
        history["train_acc"].append(float(np.mean(accs)))
        history["test_acc"].append(float(np.mean(test_accs)))
    return model, history


def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(history["loss"])
    axes[0].set_title("loss")
    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["test_acc"], label="test")
    axes[1].set_ylim(0, 1.05)
    axes[1].set_title("accuracy")
    axes[1].legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_prefix_predictions(model, name):
    _, logits_each = model.hidden_path(name.lower())
    probs = F.softmax(logits_each, dim=1).numpy()
    prefixes = [name[: i + 1].lower() for i in range(len(name))]
    plt.figure(figsize=(max(7, len(name) * 0.75), 3.5))
    for i, label in enumerate(labels):
        plt.plot(prefixes, probs[:, i], marker="o", label=label)
    plt.ylim(0, 1.05)
    plt.title(f"Prefix predictions for '{name}'")
    plt.ylabel("probability")
    plt.legend(ncol=2)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


def plot_hidden_path(model, name, max_units=14):
    hidden, _ = model.hidden_path(name.lower())
    data = hidden[:, :max_units].numpy().T
    fig, ax = plt.subplots(figsize=(max(6, len(name) * 0.7), 3.8))
    im = ax.imshow(data, aspect="auto", cmap="coolwarm")
    ax.set_xticks(range(len(name)), labels=list(name.lower()))
    ax.set_yticks(range(data.shape[0]), labels=[f"h{i}" for i in range(data.shape[0])])
    ax.set_title(f"Hidden-state path for '{name}'")
    fig.colorbar(im, ax=ax, fraction=0.025)
    plt.tight_layout()
    plt.show()


## Baseline run

Run this cell first. Do not edit the baseline config.


In [ ]:
baseline_config = {
    "embedding_dim": 12,
    "hidden_dim": 24,
    "epochs": 45,
    "lr": 0.02,
    "seed": 8,
}

baseline_model, baseline_history = train_char_rnn(baseline_config)
plot_history(baseline_history, "Baseline character RNN")
plot_prefix_predictions(baseline_model, "Kowalski")
plot_hidden_path(baseline_model, "Kowalski")
print("baseline final test accuracy:", round(baseline_history["test_acc"][-1], 3))


### Baseline notes

Write 3-5 sentences:

- What accuracy did the baseline reach?
- At which prefix did the example prediction become confident?
- What do you notice in the hidden-state heatmap?


---

## Experiment 1

Change exactly one or two settings compared with the baseline. Keep a note of what changed.


In [ ]:
# TODO: change one or two values.
experiment_1_config = {
    "embedding_dim": 8,
    "hidden_dim": 16,
    "epochs": 45,
    "lr": 0.02,
    "seed": 8,
}

experiment_1_model, experiment_1_history = train_char_rnn(experiment_1_config)
plot_history(experiment_1_history, "Experiment 1")
plot_prefix_predictions(experiment_1_model, "Schneider")
plot_hidden_path(experiment_1_model, "Schneider")
print("experiment 1 final test accuracy:", round(experiment_1_history["test_acc"][-1], 3))


### Experiment 1 notes

Write 3-5 sentences:

- What did you change?
- What happened compared with the baseline?
- Did the hidden-state path look simpler, noisier, or similar?


---

## Experiment 2

Try a different controlled change. Avoid changing everything at once.


In [ ]:
# TODO: change one or two values.
experiment_2_config = {
    "embedding_dim": 12,
    "hidden_dim": 32,
    "epochs": 35,
    "lr": 0.02,
    "seed": 8,
}

experiment_2_model, experiment_2_history = train_char_rnn(experiment_2_config)
plot_history(experiment_2_history, "Experiment 2")
plot_prefix_predictions(experiment_2_model, "Rossi")
plot_hidden_path(experiment_2_model, "Rossi")
print("experiment 2 final test accuracy:", round(experiment_2_history["test_acc"][-1], 3))


### Experiment 2 notes

Write 3-5 sentences:

- What did you change?
- Which model would you keep, and why?
- Does your choice depend only on accuracy, or also on representation behavior?


---

## Part B: attention analysis on tiny sentences

Now inspect a small attention-pooling classifier. The attention weights are useful evidence, but not perfect explanations.

### What this attention model is doing

This is single-head attention pooling, not full transformer self-attention. Each token receives a learned score; softmax turns the scores into weights that sum to 1; the model then builds a weighted average of word embeddings for classification.

If the heatmap matches your intuition, explain why. If it looks surprising, that is also useful evidence: the tiny model may be using shortcuts, especially for negation such as `not good` and `not bad`.


In [ ]:
sentence_data = [
    ("i loved the movie", 1), ("the film was excellent", 1), ("what a great story", 1),
    ("this product feels amazing", 1), ("the service was wonderful", 1), ("the lecture was clear", 1),
    ("not bad at all", 1), ("the update is surprisingly good", 1),
    ("i hated the movie", 0), ("the film was terrible", 0), ("what a boring story", 0),
    ("this product feels awful", 0), ("the service was poor", 0), ("the lecture was unclear", 0),
    ("not good at all", 0), ("the update is surprisingly broken", 0),
    ("excellent pleasant amazing movie", 1), ("bad dull useless service", 0),
]

TOKEN_RE = re.compile(r"[a-z]+")
def tokenize(text):
    return TOKEN_RE.findall(text.lower())

counter = Counter()
for text, _ in sentence_data:
    counter.update(tokenize(text))

word_to_idx = {"<PAD>": 0, "<UNK>": 1}
for word in sorted(counter):
    word_to_idx[word] = len(word_to_idx)
idx_to_word = {i: word for word, i in word_to_idx.items()}
idx_to_sentiment = {0: "negative", 1: "positive"}


def encode_sentence(text):
    return torch.tensor([word_to_idx.get(tok, word_to_idx["<UNK>"]) for tok in tokenize(text)], dtype=torch.long)


def collate_sentences(batch):
    encoded = [encode_sentence(text) for text, _ in batch]
    lengths = torch.tensor([len(x) for x in encoded], dtype=torch.long)
    y = torch.tensor([label for _, label in batch], dtype=torch.long)
    max_len = int(lengths.max())
    x = torch.zeros(len(batch), max_len, dtype=torch.long)
    for i, item in enumerate(encoded):
        x[i, : len(item)] = item
    return x, lengths, y

text_loader = DataLoader(sentence_data, batch_size=6, shuffle=True, collate_fn=collate_sentences)

class AttentionPoolingClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=12):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.attention_score = nn.Linear(embedding_dim, 1)
        self.classifier = nn.Linear(embedding_dim, 2)

    def forward(self, x, lengths, return_attention=False):
        emb = self.embedding(x)
        scores = self.attention_score(torch.tanh(emb)).squeeze(-1)
        scores = scores.masked_fill(x == 0, -1e9)
        weights = F.softmax(scores, dim=1)
        context = (emb * weights.unsqueeze(-1)).sum(dim=1)
        logits = self.classifier(context)
        if return_attention:
            return logits, weights
        return logits


def train_attention_model(epochs=160, embedding_dim=12, lr=0.025):
    set_seed(10)
    model = AttentionPoolingClassifier(len(word_to_idx), embedding_dim=embedding_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        losses, accs = [], []
        for x, lengths, y in text_loader:
            x, lengths, y = x.to(device), lengths.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x, lengths)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            accs.append(accuracy_from_logits(logits, y))
        history.append((float(np.mean(losses)), float(np.mean(accs))))
    return model, history

attention_model, attention_history = train_attention_model()
plt.figure(figsize=(6, 3))
plt.plot([x[0] for x in attention_history], label="loss")
plt.plot([x[1] for x in attention_history], label="accuracy")
plt.legend()
plt.title("Attention model training")
plt.tight_layout()
plt.show()
print("final training accuracy:", round(attention_history[-1][1], 3))


In [ ]:
@torch.no_grad()
def plot_attention(sentence, model=attention_model):
    model.eval()
    tokens = tokenize(sentence)
    x, lengths, _ = collate_sentences([(sentence, 0)])
    logits, weights = model(x.to(device), lengths.to(device), return_attention=True)
    probs = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    weights = weights.squeeze(0).cpu().numpy()[: len(tokens)]
    pred = idx_to_sentiment[int(probs.argmax())]

    fig, ax = plt.subplots(figsize=(max(6, len(tokens) * 0.8), 1.8))
    ax.imshow(weights.reshape(1, -1), cmap="YlOrRd", aspect="auto", vmin=0, vmax=max(0.45, float(weights.max())))
    ax.set_xticks(range(len(tokens)), labels=tokens, rotation=30, ha="right")
    ax.set_yticks([])
    ax.set_title(f"prediction: {pred} | p(pos)={probs[1]:.2f}")
    for i, w in enumerate(weights):
        ax.text(i, 0, f"{w:.2f}", ha="center", va="center", color="#111")
    plt.tight_layout()
    plt.show()

for sentence in ["the film was excellent", "the film was terrible", "not good at all"]:
    plot_attention(sentence)


## Your attention examples

Choose at least three sentences. Use words from the tiny vocabulary so the model has seen them before.


In [ ]:
# TODO: replace these with your own examples.
my_sentences = [
    "the service was wonderful",
    "the service was poor",
    "the movie was not good",
]

for sentence in my_sentences:
    plot_attention(sentence)


### Attention notes

Write 5-8 sentences:

- Which examples were classified confidently?
- Which token received the highest attention in each example?
- Did the attention weights match your intuition?
- Give one example where attention was helpful evidence.
- Which heatmap looked misleading, and what shortcut might the model be using?
- Give one limitation of using attention weights as explanations.


---

## Optional: embedding projection

This section is optional. It can help you inspect whether positive and negative words separate in the learned embedding space.


In [ ]:
with torch.no_grad():
    W = attention_model.embedding.weight.detach().cpu().numpy()
ids = [i for i in range(2, len(idx_to_word))]
words = [idx_to_word[i] for i in ids]
coords = PCA(n_components=2, random_state=0).fit_transform(W[ids])

plt.figure(figsize=(8, 6))
plt.scatter(coords[:, 0], coords[:, 1], color="#4c78a8")
for i, word in enumerate(words):
    plt.text(coords[i, 0] + 0.02, coords[i, 1] + 0.02, word, fontsize=9)
plt.title("Optional 2D view of attention-model embeddings")
plt.tight_layout()
plt.show()


## Final reflection

Answer in 8-12 sentences:

1. Which character-RNN configuration would you keep, and why?
2. What did the hidden-state or prefix-probability visualization show?
3. What did attention focus on in your sentence examples?
4. Where did the model behave surprisingly?
5. What is one thing these tiny models capture about sequence representations?
6. What is one thing they clearly do not capture about real language?


## Academic integrity and AI use

You may use AI assistants for debugging, explanation, and brainstorming.

You remain responsible for understanding the notebook you submit. Be prepared to explain:

- what an embedding is
- what a hidden state is
- why order matters
- what attention weights represent
- why attention weights are not a complete explanation
